# Perceptual Polynomial Straightening (3 Encoder-Decoder Pairs)

Goal: replicate the experiment structure of sushi-does-it-again.ipynb while moving polynomial fitting from latent space to perceptual encoder space in a new notebook.

Scope lock:
- New notebook only (no edits to sushi-does-it-again.ipynb).
- Same protocol: baseline run, guided run, degree sweep, curvature reporting and plots.
- Only core swap: latent polynomial fit -> perceptual feature polynomial fit.

## Paper Relevance Check for the 3 Pairs

Evidence from arXiv 2510.25420v1:
- SDXL is directly used: Section III states VISION-XL adapts pretrained SDXL [39].
- Stable Diffusion 1.5 and PixArt are explicitly listed in Section VII (Future Directions) under generalization studies (with Consistency Models also named).
- The method is training-free and intended to integrate with VISION-XL-like zero-shot pipelines, so changing diffusion backbones is relevant for transferability tests.

Decision:
1. Pair A (primary paper baseline): SDXL-style AutoencoderKL path.
2. Pair B (paper-mentioned generalization): SD 1.5 AutoencoderKL path.
3. Pair C (paper-mentioned generalization): PixArt-compatible AutoencoderKL path.

Fallback rule:
- If Pair C cannot be loaded in the runtime, replace Pair C with a Consistency-Model-compatible path (also paper-mentioned in Section VII).

In [ ]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false, reportGeneralTypeIssues=false, reportAttributeAccessIssue=false, reportCallIssue=false, reportArgumentType=false, reportUnusedImport=false, reportUnusedVariable=false
# ==========================================
# CELL 1: Configuration and Imports
# ==========================================
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from diffusers import AutoencoderKL, StableDiffusionXLPipeline, StableVideoDiffusionPipeline
from diffusers.utils import export_to_gif
from torchvision.transforms import GaussianBlur

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

CONFIG = {
    "experiment_seed": 42,
    "device": DEVICE,
    "dtype": DTYPE,
    "text_prompt": "Action shot, motion blur, a futuristic cyberpunk robot mid-stride walking down a neon-lit street, 8k resolution, photorealistic",
    "sdxl_model_id": "stabilityai/stable-diffusion-xl-base-1.0",
    "svd_model_id": "stabilityai/stable-video-diffusion-img2vid",
    "sdxl_steps": 25,
    "t2i_height": 576,
    "t2i_width": 1024,
    "height": 320,
    "width": 576,
    "num_frames": 14,
    "fps": 2,
    "decode_chunk_size": 1,
    "motion_bucket_id": 127,
    "noise_aug_strength": 0.02,
    "guidance_interval": 5,
    "refine_steps": 1,
    "guidance_lr": 0.05,
    "default_guidance_degree": 3,
    "perceptual_poly_degrees": [2, 3, 4, 5],
    "ensemble_paths": 2,
    "run_baseline": True,
    "run_guided": True,
    "run_degree_sweep": True,
    "run_mpes": True,
    "perceptual_resolution": (128, 128),
    "save_dir": "experiment-results",
}


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def reset_vram() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


set_global_seed(CONFIG["experiment_seed"])

try:
    with open("hf-access-token.txt", "r", encoding="utf-8") as f:
        HF_TOKEN = f.read().strip()
except FileNotFoundError:
    HF_TOKEN = None
    print("Warning: hf-access-token.txt not found. Public checkpoints only.")

OUTPUT_ROOT = Path(CONFIG["save_dir"])
PLOTS_DIR = OUTPUT_ROOT / "plots"
METADATA_DIR = OUTPUT_ROOT / "metadata"
for folder in [OUTPUT_ROOT, PLOTS_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PAIR_CATALOG: List[Dict[str, Any]] = [
    {
        "pair_id": "A",
        "name": "sdxl_style",
        "paper_evidence": "Section III baseline uses SDXL via VISION-XL.",
        "vae_candidates": [
            {"model_id": "madebyollin/sdxl-vae-fp16-fix", "subfolder": None},
            {"model_id": "stabilityai/stable-diffusion-xl-base-1.0", "subfolder": "vae"},
        ],
    },
    {
        "pair_id": "B",
        "name": "sd15_style",
        "paper_evidence": "Section VII future generalization mentions Stable Diffusion 1.5.",
        "vae_candidates": [
            {"model_id": "stabilityai/sd-vae-ft-mse", "subfolder": None},
            {"model_id": "runwayml/stable-diffusion-v1-5", "subfolder": "vae"},
        ],
    },
    {
        "pair_id": "C",
        "name": "pixart_style",
        "paper_evidence": "Section VII future generalization mentions PixArt (and Consistency Models).",
        "vae_candidates": [
            {"model_id": "PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers", "subfolder": "vae"},
            {"model_id": "PixArt-alpha/PixArt-XL-2-1024-MS", "subfolder": "vae"},
        ],
        "fallback_note": "If PixArt VAE cannot be loaded, keep default SVD VAE and log fallback.",
    },
]

PAIR_OUTPUT_DIRS = {
    pair["pair_id"]: OUTPUT_ROOT / f"perceptual_poly_pair_{pair['pair_id']}" for pair in PAIR_CATALOG
}
for pair_dir in PAIR_OUTPUT_DIRS.values():
    pair_dir.mkdir(parents=True, exist_ok=True)

print(f"Setup complete on {DEVICE} with dtype={DTYPE}.")
print("Active pairs:", [f"{p['pair_id']}:{p['name']}" for p in PAIR_CATALOG])

In [ ]:
# ==========================================
# CELL 2: Perceptual Encoder + Polynomial Guidance
# ==========================================
class V1PerceptualSpace(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.filters = nn.Conv2d(3, 12, kernel_size=15, padding=7, bias=False, groups=3)
        self.filters.weight.requires_grad = False
        nn.init.normal_(self.filters.weight, std=0.1)
        self.retina_blur = GaussianBlur(kernel_size=5, sigma=1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        blurred = self.retina_blur(x)
        retina_out = x - blurred
        v1_response = F.relu(self.filters(retina_out))
        pooled = F.adaptive_avg_pool2d(v1_response, (1, 1)).view(x.size(0), -1)
        return pooled


PERCEPTUAL_ENCODER = V1PerceptualSpace().to(DEVICE, DTYPE).eval()
for p in PERCEPTUAL_ENCODER.parameters():
    p.requires_grad_(False)


def preprocess_frames(frames_tensor: torch.Tensor) -> torch.Tensor:
    if frames_tensor.ndim != 4:
        raise ValueError("Expected [T, C, H, W] tensor")

    frames = frames_tensor
    if frames.max() > 1.0:
        frames = frames / 255.0

    frames = frames.to(device=DEVICE, dtype=DTYPE)
    target_h, target_w = CONFIG["perceptual_resolution"]
    if frames.shape[-2:] != (target_h, target_w):
        frames = F.interpolate(frames, size=(target_h, target_w), mode="bilinear", align_corners=False)

    return frames


def perceptual_encode(frames_tensor: torch.Tensor) -> torch.Tensor:
    processed = preprocess_frames(frames_tensor)
    return PERCEPTUAL_ENCODER(processed)


def compute_temporal_curvature(features_td: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    if features_td.ndim != 2 or features_td.size(0) < 3:
        raise ValueError("Expected [T, D] features with T >= 3")

    displacements = features_td[1:] - features_td[:-1]
    v_prev = displacements[:-1]
    v_next = displacements[1:]
    dot = (v_prev * v_next).sum(dim=-1)
    denom = (v_prev.norm(dim=-1) * v_next.norm(dim=-1)).clamp_min(eps)
    cos_theta = (dot / denom).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
    return torch.arccos(cos_theta).mean()


def fit_polynomial_features(features_td: torch.Tensor, degree: int) -> torch.Tensor:
    t = torch.linspace(-1.0, 1.0, features_td.shape[0], device=features_td.device, dtype=torch.float32)
    vandermonde = torch.vander(t, N=degree + 1, increasing=True)
    y = features_td.to(torch.float32)
    coeff = torch.linalg.pinv(vandermonde) @ y
    fitted = vandermonde @ coeff
    return fitted.to(features_td.dtype)


def decode_single_latent_frame(pipe: StableVideoDiffusionPipeline, latent_frame: torch.Tensor) -> torch.Tensor:
    scaling = float(getattr(pipe.vae.config, "scaling_factor", 0.18215))
    scaled = latent_frame / scaling

    try:
        decoded = pipe.vae.decode(scaled, num_frames=1).sample
    except TypeError:
        decoded = pipe.vae.decode(scaled).sample

    return (decoded / 2 + 0.5).clamp(0, 1)


def decode_latent_sequence_to_frames(pipe: StableVideoDiffusionPipeline, latent_seq: torch.Tensor) -> torch.Tensor:
    if latent_seq.ndim != 4:
        raise ValueError("Expected latent_seq shape [T, C, H, W]")

    decoded_frames = []
    for t_idx in range(latent_seq.shape[0]):
        frame_latent = latent_seq[t_idx : t_idx + 1]
        frame_pixel = decode_single_latent_frame(pipe, frame_latent)
        decoded_frames.append(frame_pixel)

    return torch.cat(decoded_frames, dim=0)


def tensor_frames_to_pil(frames_tensor: torch.Tensor) -> List[Image.Image]:
    frames = frames_tensor.detach().cpu().permute(0, 2, 3, 1).float().numpy()
    return [Image.fromarray((np.clip(frame, 0.0, 1.0) * 255).astype(np.uint8)) for frame in frames]


def pil_frames_to_tensor(video_frames_list: List[Image.Image]) -> torch.Tensor:
    arr = np.stack([np.array(frame).astype(np.float32) / 255.0 for frame in video_frames_list], axis=0)
    if arr.ndim == 3:
        arr = np.repeat(arr[..., None], 3, axis=-1)
    return torch.from_numpy(arr).permute(0, 3, 1, 2)


def apply_perceptual_poly_guidance(
    pipe: StableVideoDiffusionPipeline,
    latent_seq: torch.Tensor,
    degree: int,
) -> torch.Tensor:
    refine_steps = int(CONFIG["refine_steps"])
    guidance_lr = float(CONFIG["guidance_lr"])

    working = latent_seq.detach().clone()
    for _ in range(refine_steps):
        with torch.enable_grad():
            working = working.detach().requires_grad_(True)
            decoded_frames = decode_latent_sequence_to_frames(pipe, working)
            features = perceptual_encode(decoded_frames)
            target_features = fit_polynomial_features(features.detach(), degree=degree)
            loss = F.mse_loss(features.float(), target_features.float())
            grad = torch.autograd.grad(loss, working, allow_unused=False)[0]
        working = (working - guidance_lr * grad).detach()

    return working

In [ ]:
# ==========================================
# CELL 3: Callback and Model Loading Helpers
# ==========================================
CALLBACK_STATE: Dict[str, Any] = {
    "calls": 0,
    "last_error": None,
}


def _base_model_kwargs() -> Dict[str, Any]:
    kwargs: Dict[str, Any] = {
        "torch_dtype": DTYPE,
        "use_safetensors": True,
    }
    if DTYPE == torch.float16:
        kwargs["variant"] = "fp16"
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    return kwargs


def _load_vae_candidate(model_id: str, subfolder: Optional[str]) -> AutoencoderKL:
    kwargs: Dict[str, Any] = {"torch_dtype": DTYPE}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    if subfolder:
        kwargs["subfolder"] = subfolder
    return AutoencoderKL.from_pretrained(model_id, **kwargs)


def load_replacement_vae(pair_cfg: Dict[str, Any]) -> Tuple[Optional[AutoencoderKL], Dict[str, Any]]:
    meta: Dict[str, Any] = {
        "pair_name": pair_cfg["name"],
        "loaded_from": None,
        "attempts": [],
        "fallback_used": False,
    }

    for candidate in pair_cfg.get("vae_candidates", []):
        model_id = candidate["model_id"]
        subfolder = candidate.get("subfolder")
        try:
            vae = _load_vae_candidate(model_id=model_id, subfolder=subfolder)
            meta["loaded_from"] = {"model_id": model_id, "subfolder": subfolder}
            return vae.to(dtype=DTYPE), meta
        except Exception as exc:
            meta["attempts"].append({
                "model_id": model_id,
                "subfolder": subfolder,
                "error": str(exc),
            })

    meta["fallback_used"] = True
    return None, meta


def load_svd_pipeline_for_pair(pair_cfg: Dict[str, Any]) -> Tuple[StableVideoDiffusionPipeline, Dict[str, Any]]:
    pipe = StableVideoDiffusionPipeline.from_pretrained(CONFIG["svd_model_id"], **_base_model_kwargs())

    replacement_vae, vae_meta = load_replacement_vae(pair_cfg)
    if replacement_vae is not None:
        pipe.vae = replacement_vae
        vae_meta["active_vae"] = "replacement"
    else:
        vae_meta["active_vae"] = "default_svd_vae"

    if torch.cuda.is_available():
        pipe.enable_sequential_cpu_offload()
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
    else:
        pipe.to(DEVICE)

    return pipe, vae_meta


def generate_starting_frame(prompt: str) -> Image.Image:
    reset_vram()
    sdxl_pipe = StableDiffusionXLPipeline.from_pretrained(CONFIG["sdxl_model_id"], **_base_model_kwargs())

    if torch.cuda.is_available():
        sdxl_pipe.enable_model_cpu_offload()
    else:
        sdxl_pipe.to(DEVICE)

    out = sdxl_pipe(
        prompt=prompt,
        num_inference_steps=CONFIG["sdxl_steps"],
        height=CONFIG["t2i_height"],
        width=CONFIG["t2i_width"],
    )
    generated_image = out.images[0]

    del sdxl_pipe
    reset_vram()

    return generated_image.resize((CONFIG["width"], CONFIG["height"]))


def build_perceptual_guidance_callback(pipe: StableVideoDiffusionPipeline, degree: int):
    def _callback(_pipeline, step_index, _timestep, callback_kwargs):
        _ = (_pipeline, _timestep)
        CALLBACK_STATE["calls"] += 1

        if step_index % int(CONFIG["guidance_interval"]) != 0:
            return callback_kwargs

        latents = callback_kwargs["latents"]
        try:
            seq = latents[0].permute(1, 0, 2, 3)
            refined_seq = apply_perceptual_poly_guidance(pipe, seq, degree=degree)
            latents = latents.clone()
            latents[0] = refined_seq.permute(1, 0, 2, 3)
            callback_kwargs["latents"] = latents
        except Exception as exc:
            CALLBACK_STATE["last_error"] = str(exc)

        return callback_kwargs

    return _callback

In [ ]:
# ==========================================
# CELL 4: Per-Pair Experiment Execution
# ==========================================
def run_video_generation(
    pipe: StableVideoDiffusionPipeline,
    input_img_pil: Image.Image,
    seed: int,
    callback=None,
    output_type: str = "pil",
):
    gen_kwargs = {
        "decode_chunk_size": CONFIG["decode_chunk_size"],
        "generator": torch.manual_seed(seed),
        "motion_bucket_id": CONFIG["motion_bucket_id"],
        "noise_aug_strength": CONFIG["noise_aug_strength"],
        "num_frames": CONFIG["num_frames"],
        "output_type": output_type,
    }
    if callback is not None:
        gen_kwargs["callback_on_step_end"] = callback
        gen_kwargs["callback_on_step_end_tensor_inputs"] = ["latents"]

    result = pipe(input_img_pil, **gen_kwargs)
    if output_type == "latent":
        return result.frames
    return result.frames[0]


def compute_curvature_from_pil(video_frames_list: List[Image.Image]) -> float:
    frame_tensor = pil_frames_to_tensor(video_frames_list)
    with torch.no_grad():
        features = perceptual_encode(frame_tensor)
        curvature = compute_temporal_curvature(features)
    return float(curvature.detach().cpu())


def mpes_generation(
    pipe: StableVideoDiffusionPipeline,
    input_img_pil: Image.Image,
    paths: int,
    base_seed: int,
) -> List[Image.Image]:
    latent_runs = []
    for path_idx in range(paths):
        latents = run_video_generation(
            pipe=pipe,
            input_img_pil=input_img_pil,
            seed=base_seed + path_idx,
            callback=None,
            output_type="latent",
        )
        if isinstance(latents, list):
            latents = latents[0]
        latent_runs.append(latents)

    avg_latents = torch.stack(latent_runs, dim=0).mean(dim=0)

    if avg_latents.ndim == 5:
        seq = avg_latents[0]
    elif avg_latents.ndim == 4:
        seq = avg_latents
    else:
        raise RuntimeError(f"Unexpected latent shape: {tuple(avg_latents.shape)}")

    # Convert [C, F, H, W] to [F, C, H, W] when needed.
    if seq.shape[0] <= 4 and seq.shape[1] > 4:
        seq = seq.permute(1, 0, 2, 3)

    decoded = decode_latent_sequence_to_frames(pipe, seq)
    return tensor_frames_to_pil(decoded)


def plot_pair_curvature(pair_metrics: Dict[str, Any], save_path: Path) -> None:
    baseline_curv = pair_metrics.get("runs", {}).get("baseline", {}).get("curvature")
    guided_curv = pair_metrics.get("runs", {}).get("guided", {}).get("curvature")
    degree_curv = pair_metrics.get("runs", {}).get("degree_sweep", {})

    fig = plt.figure(figsize=(11, 4))

    ax1 = fig.add_subplot(1, 2, 1)
    labels = ["baseline", "guided"]
    vals = [
        np.nan if baseline_curv is None else baseline_curv,
        np.nan if guided_curv is None else guided_curv,
    ]
    ax1.bar(labels, vals, color=["#b23a48", "#2a9d8f"])
    ax1.set_title(f"Pair {pair_metrics['pair_id']} Curvature")
    ax1.set_ylabel("Curvature (radians)")

    ax2 = fig.add_subplot(1, 2, 2)
    deg_keys = sorted([int(k) for k in degree_curv.keys()])
    deg_vals = [degree_curv[str(k)]["curvature"] for k in deg_keys]
    if deg_keys:
        ax2.plot(deg_keys, deg_vals, marker="o", color="#264653")
    ax2.set_title(f"Pair {pair_metrics['pair_id']} Degree Sweep")
    ax2.set_xlabel("Polynomial degree")
    ax2.set_ylabel("Curvature (radians)")
    ax2.set_xticks(CONFIG["perceptual_poly_degrees"])

    fig.tight_layout()
    fig.savefig(str(save_path))
    plt.close(fig)


def plot_combined_summary(all_pair_metrics: Dict[str, Dict[str, Any]], save_path: Path) -> None:
    pair_ids = []
    baseline_vals = []
    guided_vals = []

    for pair_id, payload in all_pair_metrics.items():
        pair_ids.append(pair_id)
        baseline_vals.append(payload.get("runs", {}).get("baseline", {}).get("curvature", np.nan))
        guided_vals.append(payload.get("runs", {}).get("guided", {}).get("curvature", np.nan))

    x = np.arange(len(pair_ids))
    width = 0.35

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(1, 1, 1)
    ax.bar(x - width / 2, baseline_vals, width, label="Baseline", color="#e76f51")
    ax.bar(x + width / 2, guided_vals, width, label="Guided", color="#2a9d8f")
    ax.set_xticks(x)
    ax.set_xticklabels(pair_ids)
    ax.set_xlabel("Encoder-Decoder Pair")
    ax.set_ylabel("Curvature (radians)")
    ax.set_title("Baseline vs Guided Curvature Across Pairs")
    ax.legend()

    fig.tight_layout()
    fig.savefig(str(save_path))
    plt.close(fig)


print("Generating SDXL starting frame...")
starting_frame = generate_starting_frame(CONFIG["text_prompt"])
starting_frame_path = OUTPUT_ROOT / "starting_frame.png"
starting_frame.save(starting_frame_path)
print(f"Saved starting frame to: {starting_frame_path}")

all_pair_metrics: Dict[str, Dict[str, Any]] = {}

for pair_cfg in PAIR_CATALOG:
    pair_id = pair_cfg["pair_id"]
    pair_name = pair_cfg["name"]
    pair_dir = PAIR_OUTPUT_DIRS[pair_id]

    pair_metrics: Dict[str, Any] = {
        "pair_id": pair_id,
        "pair_name": pair_name,
        "paper_evidence": pair_cfg["paper_evidence"],
        "vae_meta": {},
        "runs": {
            "baseline": {},
            "guided": {},
            "degree_sweep": {},
            "mpes": {},
        },
    }

    pipe: Optional[StableVideoDiffusionPipeline] = None
    print(f"\n=== Running Pair {pair_id}: {pair_name} ===")

    try:
        pipe, vae_meta = load_svd_pipeline_for_pair(pair_cfg)
        pair_metrics["vae_meta"] = vae_meta

        if CONFIG["run_baseline"]:
            baseline_frames = run_video_generation(
                pipe=pipe,
                input_img_pil=starting_frame,
                seed=CONFIG["experiment_seed"],
                callback=None,
            )
            baseline_path = pair_dir / "baseline.gif"
            export_to_gif(baseline_frames, baseline_path, fps=CONFIG["fps"])
            baseline_curv = compute_curvature_from_pil(baseline_frames)
            pair_metrics["runs"]["baseline"] = {
                "gif_path": str(baseline_path),
                "curvature": baseline_curv,
            }
            print(f"Baseline curvature: {baseline_curv:.5f}")

        if CONFIG["run_guided"]:
            guided_callback = build_perceptual_guidance_callback(
                pipe=pipe,
                degree=CONFIG["default_guidance_degree"],
            )
            guided_frames = run_video_generation(
                pipe=pipe,
                input_img_pil=starting_frame,
                seed=CONFIG["experiment_seed"],
                callback=guided_callback,
            )
            guided_path = pair_dir / "guided.gif"
            export_to_gif(guided_frames, guided_path, fps=CONFIG["fps"])
            guided_curv = compute_curvature_from_pil(guided_frames)
            pair_metrics["runs"]["guided"] = {
                "gif_path": str(guided_path),
                "curvature": guided_curv,
                "callback_error": CALLBACK_STATE["last_error"],
            }
            print(f"Guided curvature: {guided_curv:.5f}")

        if CONFIG["run_degree_sweep"]:
            for degree in CONFIG["perceptual_poly_degrees"]:
                degree_callback = build_perceptual_guidance_callback(pipe=pipe, degree=degree)
                degree_frames = run_video_generation(
                    pipe=pipe,
                    input_img_pil=starting_frame,
                    seed=CONFIG["experiment_seed"],
                    callback=degree_callback,
                )
                degree_path = pair_dir / f"guided_degree_{degree}.gif"
                export_to_gif(degree_frames, degree_path, fps=CONFIG["fps"])
                degree_curv = compute_curvature_from_pil(degree_frames)
                pair_metrics["runs"]["degree_sweep"][str(degree)] = {
                    "gif_path": str(degree_path),
                    "curvature": degree_curv,
                }
                print(f"Degree {degree} curvature: {degree_curv:.5f}")
                del degree_frames
                reset_vram()

        if CONFIG["run_mpes"]:
            mpes_frames = mpes_generation(
                pipe=pipe,
                input_img_pil=starting_frame,
                paths=CONFIG["ensemble_paths"],
                base_seed=CONFIG["experiment_seed"],
            )
            mpes_path = pair_dir / "mpes.gif"
            export_to_gif(mpes_frames, mpes_path, fps=CONFIG["fps"])
            mpes_curv = compute_curvature_from_pil(mpes_frames)
            pair_metrics["runs"]["mpes"] = {
                "gif_path": str(mpes_path),
                "curvature": mpes_curv,
                "paths": CONFIG["ensemble_paths"],
            }
            print(f"MPES curvature: {mpes_curv:.5f}")

        pair_plot_path = PLOTS_DIR / f"pair_{pair_id}_curvature.png"
        plot_pair_curvature(pair_metrics, pair_plot_path)
        pair_metrics["pair_plot"] = str(pair_plot_path)

    except Exception as exc:
        pair_metrics["error"] = str(exc)
        print(f"Pair {pair_id} failed: {exc}")

    finally:
        pair_metrics_path = METADATA_DIR / f"metrics_pair_{pair_id}.json"
        with open(pair_metrics_path, "w", encoding="utf-8") as f:
            json.dump(pair_metrics, f, indent=2)
        all_pair_metrics[pair_id] = pair_metrics

        if pipe is not None:
            del pipe
        reset_vram()

summary_plot_path = PLOTS_DIR / "curvature_summary_across_pairs.png"
plot_combined_summary(all_pair_metrics, summary_plot_path)

all_metrics_path = METADATA_DIR / "all_pair_metrics.json"
with open(all_metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_pair_metrics, f, indent=2)

print("\nExperiment orchestration complete.")
print(f"Combined plot: {summary_plot_path}")
print(f"Metrics JSON:  {all_metrics_path}")
all_pair_metrics

## Cell 7: Evaluation and Verification Checklist

Metrics and plots (same style as sushi):
- Baseline curvature vs guided curvature per pair.
- Degree-wise curvature trend per pair for degrees [2, 3, 4, 5].
- One comparison plot per pair.
- One combined summary plot across all 3 pairs.

Output layout:
- experiment-results/perceptual_poly_pair_A
- experiment-results/perceptual_poly_pair_B
- experiment-results/perceptual_poly_pair_C
- experiment-results/plots
- experiment-results/metadata

Verification gates:
1. Protocol parity with sushi (seeds, degree list, cadence, metrics).
2. Polynomial fitting operates on perceptual features, not latent tensors.
3. All three pairs produce baseline, guided, and degree outputs.
4. Curvature values are finite for all runs.
5. Runtime remains memory-safe for RTX 4060-class hardware.

## Additional Evaluation Metrics and Comparison Plots

This section adds:
- Temporal metrics from hypothesis_testing.ipynb: SSIM (higher is better) and PTD (lower is better)
- Trajectory comparison plots inspired by doing_it_right.ipynb using 2D PCA-style projection and straightness scores

In [ ]:
# ==========================================
# Additional Metrics + Comparison Plots
# ==========================================
import glob
import importlib
import json
import os

np = importlib.import_module("numpy")
plt = importlib.import_module("matplotlib.pyplot")


def read_gif_frames(gif_path: str):
    pil_image = importlib.import_module("PIL.Image")
    pil_sequence = importlib.import_module("PIL.ImageSequence")
    with pil_image.open(gif_path) as img:
        return [frame.convert("RGB").copy() for frame in pil_sequence.Iterator(img)]


def pca_2d(features):
    centered = features - features.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    return centered @ vh[:2].T


def calculate_straightness(trajectory):
    if len(trajectory) < 3:
        return 0.0
    deltas = trajectory[1:] - trajectory[:-1]
    norms = np.linalg.norm(deltas, axis=1, keepdims=True)
    deltas_norm = deltas / (norms + 1e-8)
    cos_sims = np.sum(deltas_norm[:-1] * deltas_norm[1:], axis=1)
    return float(np.mean(cos_sims))


def _load_optional_ssim_callable():
    try:
        metrics_mod = importlib.import_module("skimage.metrics")
        return getattr(metrics_mod, "structural_similarity", None)
    except Exception:
        return None


def _simple_ssim_rgb(a, b) -> float:
    # Fallback global SSIM approximation if skimage is unavailable.
    if a.ndim == 3:
        a_gray = 0.2989 * a[..., 0] + 0.5870 * a[..., 1] + 0.1140 * a[..., 2]
        b_gray = 0.2989 * b[..., 0] + 0.5870 * b[..., 1] + 0.1140 * b[..., 2]
    else:
        a_gray = a
        b_gray = b

    mu_a = float(np.mean(a_gray))
    mu_b = float(np.mean(b_gray))
    var_a = float(np.var(a_gray))
    var_b = float(np.var(b_gray))
    cov_ab = float(np.mean((a_gray - mu_a) * (b_gray - mu_b)))

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    num = (2.0 * mu_a * mu_b + c1) * (2.0 * cov_ab + c2)
    den = (mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2) + 1e-12
    return float(num / den)


def _num(value) -> float:
    try:
        return np.asarray(value, dtype=np.float64).item()
    except Exception:
        return float("nan")


def compute_temporal_metrics(frames):
    frames_np = [np.asarray(frame).astype(np.float32) / 255.0 for frame in frames]
    ssim_callable = _load_optional_ssim_callable()

    ssim_vals = []
    ptd_vals = []
    for i in range(len(frames_np) - 1):
        a = frames_np[i]
        b = frames_np[i + 1]
        ptd_vals.append(_num(np.mean(np.abs(a - b)) * 255.0))

        if callable(ssim_callable):
            ssim_vals.append(_num(ssim_callable(a, b, channel_axis=2, data_range=1.0)))
        else:
            ssim_vals.append(_simple_ssim_rgb(a, b))

    frame_vectors = np.stack([x.reshape(-1) for x in frames_np], axis=0)
    trajectory_2d = pca_2d(frame_vectors)
    straightness = calculate_straightness(trajectory_2d)

    return {
        "mean_ssim": _num(np.mean(ssim_vals)) if ssim_vals else float("nan"),
        "mean_ptd": _num(np.mean(ptd_vals)) if ptd_vals else float("nan"),
        "straightness": _num(straightness),
        "trajectory_2d": trajectory_2d,
    }


save_dir = CONFIG["save_dir"] if "CONFIG" in globals() else "experiment-results"
plots_dir = os.path.join(save_dir, "plots")
metadata_dir = os.path.join(save_dir, "metadata")
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)

pair_dirs = sorted(glob.glob(os.path.join(save_dir, "perceptual_poly_pair_*")))

rows = []
pair_trajs = {}

for pair_dir in pair_dirs:
    pair_name = os.path.basename(pair_dir).replace("perceptual_poly_pair_", "")

    candidates = {
        "baseline": os.path.join(pair_dir, "baseline.gif"),
        "guided": os.path.join(pair_dir, "guided.gif"),
        "mpes": os.path.join(pair_dir, "mpes.gif"),
    }

    for degree_path in sorted(glob.glob(os.path.join(pair_dir, "guided_degree_*.gif"))):
        degree_name = os.path.splitext(os.path.basename(degree_path))[0]
        candidates[degree_name] = degree_path

    for method, path in candidates.items():
        if not os.path.exists(path):
            continue

        frames = read_gif_frames(path)
        if len(frames) < 3:
            continue

        metrics = compute_temporal_metrics(frames)
        rows.append(
            {
                "pair": pair_name,
                "method": method,
                "mean_ssim": _num(metrics["mean_ssim"]),
                "mean_ptd": _num(metrics["mean_ptd"]),
                "straightness": _num(metrics["straightness"]),
                "num_frames": int(len(frames)),
                "source_path": path,
            }
        )

        if method in {"baseline", "guided"}:
            pair_trajs.setdefault(pair_name, {})[method] = metrics["trajectory_2d"]

if not rows:
    print("No pair GIFs found yet. Run the generation cell first, then re-run this cell.")
else:
    print("\nPair-wise metrics (Hypothesis-Testing style):")
    print(f"{'Pair':<8}{'Method':<18}{'SSIM':>10}{'PTD':>10}{'Straight':>12}")
    for row in rows:
        ssim_value = _num(row["mean_ssim"])
        ptd_value = _num(row["mean_ptd"])
        straight_value = _num(row["straightness"])
        print(f"{row['pair']:<8}{row['method']:<18}{ssim_value:>10.4f}{ptd_value:>10.3f}{straight_value:>12.4f}")

    metrics_json = os.path.join(metadata_dir, "additional_eval_metrics_perceptual_pairs.json")
    with open(metrics_json, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)

    pair_ids = sorted({r["pair"] for r in rows})
    baseline_ssim, guided_ssim = [], []
    baseline_ptd, guided_ptd = [], []

    for pair_id in pair_ids:
        base_row = next((r for r in rows if r["pair"] == pair_id and r["method"] == "baseline"), None)
        guided_row = next((r for r in rows if r["pair"] == pair_id and r["method"] == "guided"), None)

        baseline_ssim.append(float("nan") if base_row is None else _num(base_row["mean_ssim"]))
        guided_ssim.append(float("nan") if guided_row is None else _num(guided_row["mean_ssim"]))
        baseline_ptd.append(float("nan") if base_row is None else _num(base_row["mean_ptd"]))
        guided_ptd.append(float("nan") if guided_row is None else _num(guided_row["mean_ptd"]))

    x = np.arange(len(pair_ids))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(x - width / 2, baseline_ssim, width, label="Baseline", color="#2a9d8f")
    axes[0].bar(x + width / 2, guided_ssim, width, label="Guided", color="#264653")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(pair_ids)
    axes[0].set_title("Per-Pair Temporal SSIM")
    axes[0].set_ylabel("Mean SSIM")
    axes[0].legend()

    axes[1].bar(x - width / 2, baseline_ptd, width, label="Baseline", color="#e76f51")
    axes[1].bar(x + width / 2, guided_ptd, width, label="Guided", color="#f4a261")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(pair_ids)
    axes[1].set_title("Per-Pair Temporal PTD")
    axes[1].set_ylabel("Mean PTD")
    axes[1].legend()

    fig.tight_layout()
    metrics_plot_path = os.path.join(plots_dir, "additional_pair_metrics_comparison.png")
    fig.savefig(metrics_plot_path, dpi=150)
    plt.show()
    plt.close(fig)

    valid_pairs = [p for p in pair_ids if p in pair_trajs and len(pair_trajs[p]) > 0]
    if valid_pairs:
        fig, axes = plt.subplots(1, len(valid_pairs), figsize=(5 * len(valid_pairs), 4), squeeze=False)
        for idx, pair_id in enumerate(valid_pairs):
            ax = axes[0, idx]
            for method, color in [("baseline", "#e76f51"), ("guided", "#2a9d8f")]:
                traj = pair_trajs[pair_id].get(method)
                if traj is None:
                    continue
                ax.plot(traj[:, 0], traj[:, 1], marker="o", linewidth=1.7, alpha=0.9, color=color, label=method)
                ax.scatter(traj[0, 0], traj[0, 1], marker="x", s=60, color="black")
                ax.scatter(traj[-1, 0], traj[-1, 1], marker="*", s=80, color="black")
            ax.set_title(f"Pair {pair_id} Trajectory")
            ax.set_xlabel("Component 1")
            ax.set_ylabel("Component 2")
            ax.grid(True, alpha=0.3)
            ax.legend()

        fig.tight_layout()
        traj_plot_path = os.path.join(plots_dir, "additional_pair_trajectory_comparison.png")
        fig.savefig(traj_plot_path, dpi=150)
        plt.show()
        plt.close(fig)
    else:
        traj_plot_path = None
        print("No baseline/guided trajectories available for pair comparison plots.")

    print(f"\nSaved metrics JSON: {metrics_json}")
    print(f"Saved metrics plot: {metrics_plot_path}")
    if traj_plot_path:
        print(f"Saved trajectory plot: {traj_plot_path}")